In [ ]:
pip install gensim scikit-learn nltk


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 34.7 MB/s eta 0:00:00


In [ ]:
import os
from typing import List, Tuple, Dict

import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.datasets import fetch_20newsgroups

from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel
from gensim.models.phrases import Phrases, Phraser
from gensim.parsing.preprocessing import STOPWORDS
from gensim.utils import simple_preprocess


# ===============================
# Configuration
# ===============================

N_DOCS = 1000
NUM_TOPICS = 10
PASSES = 15
MODELS_DIR = "models"

CUSTOM_STOPWORDS = {
    "people","know","like","time","said","think","use",
    "thanks","year","want","good",
    "edu","com","subject","organization","writes",
    "article","lines","from","re",
}


# ===============================
# LDA Topic Model Class
# ===============================

class LDATopicModel:

    def __init__(self):
        self.lda = None
        self.dictionary = None
        self.bigram = None

        nltk.download("wordnet", quiet=True)
        nltk.download("omw-1.4", quiet=True)

        self.lemmatizer = WordNetLemmatizer()

    # --------------------------------
    # Preprocessing
    # --------------------------------
    def preprocess(self, text: str) -> List[str]:
        tokens = simple_preprocess(text, deacc=True, min_len=3, max_len=20)
        stop = set(STOPWORDS) | CUSTOM_STOPWORDS
        tokens = [t for t in tokens if t not in stop]
        tokens = [self.lemmatizer.lemmatize(t) for t in tokens]
        return tokens

    def build_bigrams(self, docs: List[List[str]]) -> List[List[str]]:
        phrases = Phrases(docs, min_count=5, threshold=10)
        self.bigram = Phraser(phrases)
        return [self.bigram[d] for d in docs]

    # --------------------------------
    # Training
    # --------------------------------
    def train(self, raw_documents: List[str]) -> None:

        print("Preprocessing...")
        docs = [self.preprocess(d) for d in raw_documents]
        docs = self.build_bigrams(docs)

        print("Building dictionary...")
        self.dictionary = Dictionary(docs)
        self.dictionary.filter_extremes(no_below=5, no_above=0.5)
        self.dictionary.compactify()

        corpus = [self.dictionary.doc2bow(d) for d in docs]

        print("Training LDA model...")
        self.lda = LdaModel(
            corpus=corpus,
            id2word=self.dictionary,
            num_topics=NUM_TOPICS,
            passes=PASSES,
            alpha="auto",
            eta="auto",
            random_state=42,
            eval_every=None,
        )

        # Coherence
        coherence_model = CoherenceModel(
            model=self.lda,
            texts=docs,
            dictionary=self.dictionary,
            coherence="c_v"
        )
        coherence = coherence_model.get_coherence()
        print(f"\nCoherence Score: {coherence:.4f}")

        self.save()

        print("\n=== Topics ===")
        for tid, topic in self.lda.print_topics(NUM_TOPICS, 10):
            print(f"Topic {tid}: {topic}")

    # --------------------------------
    # Save / Load
    # --------------------------------
    def save(self):
        os.makedirs(MODELS_DIR, exist_ok=True)
        self.lda.save(os.path.join(MODELS_DIR, "lda_model"))
        self.dictionary.save(os.path.join(MODELS_DIR, "dictionary.dict"))

    def load(self):
        self.lda = LdaModel.load(os.path.join(MODELS_DIR, "lda_model"))
        self.dictionary = Dictionary.load(os.path.join(MODELS_DIR, "dictionary.dict"))

    # --------------------------------
    # Topic Utilities
    # --------------------------------
    def auto_label(self, topic_id: int, topn: int = 3) -> str:
        words = [w for w, _ in self.lda.show_topic(topic_id, topn=topn)]
        return "_".join(words)

    def classify(self, text: str, top_k: int = 3) -> List[Tuple[int, float]]:
        tokens = self.preprocess(text)
        if self.bigram:
            tokens = self.bigram[tokens]

        bow = self.dictionary.doc2bow(tokens)
        topics = self.lda.get_document_topics(bow, minimum_probability=0.0)
        topics = sorted(topics, key=lambda x: x[1], reverse=True)
        return topics[:top_k]


# ===============================
# Main Execution
# ===============================

def main():

    print("Loading dataset...")
    data = fetch_20newsgroups(
        subset="train",
        remove=("headers","footers","quotes")
    )

    documents = data.data[:N_DOCS]

    model = LDATopicModel()

    # Train
    model.train(documents)

    # Load model again (simulation of production usage)
    model.load()

    print("\n=== Automatic Topic Labels ===")
    labels: Dict[int, str] = {}
    for i in range(NUM_TOPICS):
        label = model.auto_label(i)
        labels[i] = label
        print(f"{i}: {label}")

    # Inference examples
    samples = [
        "The GPU delivers amazing gaming performance with high frame rates.",
        "NASA scientists discovered a new exoplanet in deep space.",
        "The basketball team won the championship after a dramatic final.",
        "The government passed a new healthcare reform bill.",
        "Italian pasta and pizza are popular traditional dishes."
    ]

    print("\n=== Inference Results ===")

    for text in samples:
        print("\n" + "="*60)
        print("Text:", text)

        results = model.classify(text)

        for tid, prob in results:
            print(f"Topic {tid} ({labels[tid]}) -> {prob:.4f}")


if __name__ == "__main__":
    main()


Loading dataset...
Preprocessing...
Building dictionary...
Training LDA model...

Coherence Score: 0.3894

=== Topics ===
Topic 0: 0.019*"armenian" + 0.008*"government" + 0.007*"greek" + 0.006*"state" + 0.006*"case" + 0.006*"turkish" + 0.005*"killed" + 0.005*"health" + 0.005*"source" + 0.005*"person"
Topic 1: 0.013*"space" + 0.011*"mission" + 0.009*"shuttle" + 0.007*"orbit" + 0.007*"nasa" + 0.007*"launch" + 0.006*"pitcher" + 0.006*"flight" + 0.006*"bike" + 0.006*"satellite"
Topic 2: 0.017*"jesus" + 0.013*"god" + 0.009*"argument" + 0.008*"thing" + 0.007*"way" + 0.007*"christian" + 0.007*"matthew" + 0.006*"true" + 0.006*"come" + 0.006*"father"
Topic 3: 0.008*"nasa" + 0.007*"application" + 0.007*"program" + 0.007*"information" + 0.006*"member" + 0.006*"muslim" + 0.006*"government" + 0.006*"armenian" + 0.005*"degree" + 0.005*"security"
Topic 4: 0.010*"read" + 0.009*"problem" + 0.008*"book" + 0.007*"new" + 0.007*"address" + 0.007*"israel" + 0.007*"error" + 0.006*"need" + 0.006*"battery" + 0